In [1]:
# Cell B1 — Tab B setup (SAC training tab).
# Confirms we're on L4/T4 (NOT H100 — we save H100 for TD-MPC2),
# mounts the SAME Drive folder Tab A uses, and binds ROOT + sys.path.

!nvidia-smi | grep "Tesla\|L4\|A100\|H100"

from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/tdmpc2-highway')
assert ROOT.exists(), f"Drive not mounted properly — {ROOT} missing"
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"\nWorking dir: {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")
print(f"\nIMPORTANT: Tab B should be on L4 or T4, not H100/A100.")
print(f"           If you see H100 above, switch runtime now.")

|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Working dir: /content/drive/MyDrive/tdmpc2-highway
sys.path[0]: /content/drive/MyDrive/tdmpc2-highway

IMPORTANT: Tab B should be on L4 or T4, not H100/A100.
           If you see H100 above, switch runtime now.


In [2]:
# Cell B2 — Install Python deps. Same set as Tab A.
# numpy<2 pin: SB3 still trips on the NumPy 2.x C-ABI break.

%%capture install_log
!pip install -q \
    "numpy<2" \
    "torch>=2.1" \
    "gymnasium>=0.29" \
    "highway-env>=1.8" \
    "stable-baselines3[extra]>=2.2" \
    termcolor wandb \
    matplotlib pandas seaborn tqdm pyyaml

In [3]:
print("Install complete. After this cell, restart the runtime once for the numpy<2 pin to take effect.")
print("Then re-run Cell B1 to remount Drive, then continue from Cell B3.")

Install complete. After this cell, restart the runtime once for the numpy<2 pin to take effect.
Then re-run Cell B1 to remount Drive, then continue from Cell B3.


In [4]:
# Cell B3 — Verify the environment, the existing src modules, and log into wandb.
import numpy as np, torch
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")

print(f"NumPy: {np.__version__}")
print(f"Torch: {torch.__version__}")
print(f"GPU:   {torch.cuda.get_device_name(0)}")
assert np.__version__.startswith("1."), "NumPy<2 pin failed"
assert torch.cuda.is_available(), "No GPU"

# Verify src modules from Tab A are importable
from src.envs.highway_factory import make_highway_env
from src.training.train_sac import train_sac
from src.utils.config import SEEDS, TOTAL_TIMESTEPS, TRAIN_ENV_ID
print(f"\n[ok] src modules importable")
print(f"[ok] SEEDS={SEEDS}, TOTAL_TIMESTEPS={TOTAL_TIMESTEPS:,}")

# Quick env smoke-test (no training, just a reset)
env = make_highway_env(env_id="highway-v0", seed=0)
obs, info = env.reset(seed=0)
print(f"[ok] env loads: obs={obs.shape}, action={env.action_space.shape}")
env.close()

# wandb login
import wandb
wandb.login()
print("\n[ok] Tab B ready for SAC training")

NumPy: 1.26.4
Torch: 2.10.0+cu128
GPU:   NVIDIA L4


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



[ok] src modules importable
[ok] SEEDS=[0, 1, 2], TOTAL_TIMESTEPS=100,000
[ok] env loads: obs=(25,), action=(2,)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
/usr/local/lib/python3.12/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
wandb: 

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: error1rak (error1rak-american-university-of-sharjah) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



[ok] Tab B ready for SAC training


In [5]:
# B3.1 — Force-install numpy<2. The %%capture in B2 may have hidden a failure.
# This time we surface the output so we can see what's happening.
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy<2"],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout[-500:] if result.stdout else "(empty)")
print("STDERR:", result.stderr[-500:] if result.stderr else "(empty)")
print(f"\nreturncode: {result.returncode}")
if result.returncode != 0:
    raise SystemExit("Install failed — see stderr above")
print("[ok] numpy<2 force-installed")

STDOUT: (empty)
STDERR: mpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


returncode: 0
[ok] numpy<2 force-installed


In [6]:
# Cell B4 — Tiny SAC run on this tab to confirm pipeline works on L4/T4.
# Already verified on Tab A's H100; running here too just to be safe.
import importlib
import src.training.train_sac
importlib.reload(src.training.train_sac)
from src.training.train_sac import train_sac

_ = train_sac(seed=99, total_timesteps=2_000, use_wandb=False)
print("\n[ok] SAC pipeline functional on Tab B")

Using cuda device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Logging to /content/drive/MyDrive/tdmpc2-highway/checkpoints/sac_highway-v0_seed99/tb/SAC_2


Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

...

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[SAC seed=99] saved final → /content/drive/MyDrive/tdmpc2-highway/checkpoints/sac_highway-v0_seed99/final.zip

[ok] SAC pipeline functional on Tab B


In [ ]:
# Cell B5 — 🔴 LONG TRAIN: SAC × 3 seeds.
# L4 estimate:  ~25-30 min/seed × 3 ≈ 75-90 min total.
# T4 estimate:  ~40-50 min/seed × 3 ≈ 2-2.5 hours total.
import importlib
import src.training.train_sac
importlib.reload(src.training.train_sac)
from src.training.train_sac import train_sac
from src.utils.config import SEEDS

print(f"Plan: SAC × {len(SEEDS)} seeds = {SEEDS}")
print(f"      Each seed: 100k steps on highway-v0\n")

sac_paths = []
for seed in SEEDS:
    print(f"\n{'='*60}\n  SAC seed {seed}\n{'='*60}")
    p = train_sac(seed=seed, use_wandb=True)
    sac_paths.append(p)

print("\n" + "="*60)
print("All SAC seeds complete:")
for p in sac_paths:
    print(f"  {p}")
print("="*60)

Output hidden; open in https://colab.research.google.com to view.